# serialize

> The .ipynb file boundary: serialize the in-memory `nb` (see notebook.py) to a real Jupyter notebook and load one back, round-tripping cell types, visibility, output blocks, assistant `details`, and the nbdev `#| export` pragma. Kept apart from both the data model (notebook.py) and the rendering/routes (cells.py) so the file-format concerns live in one place.

In [ ]:
#| default_exp serialize

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path
import nbformat as _nbf
from boopiter.notebook import nb, Notebook, CTYPES
from boopiter.kernel import _MIME_PRIORITY

## The `#| export` pragma

boopiter keeps nbdev's `#| export` pragma out of a cell's editable source entirely, tracking it as a flag instead. `_has_export`/`_strip_export` detect and remove that leading line when reading a cell from disk.

In [ ]:
#| export
# nbdev's '#| export' pragma is a leading line in a code cell's on-disk source. We keep it OUT
# of c.source (and the editor) entirely, tracking it instead as Cell.export (a plain bool) --
# these two helpers are only needed at the load_notebook()/save_notebook() file boundary.
def _has_export(source:str) -> bool:
    "True if `source`'s first line is (some spacing variant of) the '#| export' pragma."
    return source.split('\n', 1)[0].strip().replace(' ', '') == '#|export'

In [ ]:
#| export
def _strip_export(source:str) -> str:
    'The source with any leading #| export pragma line removed.'
    if not _has_export(source): return source
    rest = source.split('\n', 1)
    return rest[1] if len(rest) > 1 else ''

## Saving

`save_notebook` writes `nb` out as a real Jupyter `.ipynb`. `_blocks_to_nb_outputs` converts boopiter's output blocks into valid nbformat outputs (so saved files still render in Jupyter), and `_BOOP2NB` maps boopiter cell types onto nbformat ones.

In [ ]:
#| export
_BOOP2NB = {'code':'code', 'note':'markdown', 'prompt':'markdown', 'raw':'raw', 'assistant':'markdown'}  # our ctype -> nbformat cell_type

In [ ]:
#| export
def _blocks_to_nb_outputs(blocks:list[dict]) -> list:
    "Cell.output's block list -> real nbformat outputs, so saved .ipynb files stay valid (and render in GitHub/real Jupyter too)."
    outs = []
    for b in blocks:
        if b['type'] == 'stream':
            outs.append(_nbf.v4.new_output('stream', name='stdout', text=b['data']))
        elif b['type'] == 'error':
            ename, _, evalue = b['data'].partition(': ')
            outs.append(_nbf.v4.new_output('error', ename=ename, evalue=evalue, traceback=[b['data']]))
        else:  # display
            outs.append(_nbf.v4.new_output('display_data', data={b['mime']: b['data']}))
    return outs

def _nb_outputs_to_blocks(outputs:list) -> list[dict]:
    "Inverse of _blocks_to_nb_outputs()."
    blocks = []
    for o in outputs:
        ot = o.get('output_type')
        if ot == 'stream':
            blocks.append({'type':'stream', 'mime':None, 'data':o.get('text', '')})
        elif ot == 'error':
            data = f"{o.get('ename','')}: {o.get('evalue','')}" if o.get('ename') else o.get('evalue', '')
            blocks.append({'type':'error', 'mime':None, 'data':data})
        elif ot in ('display_data', 'execute_result'):
            data = o.get('data', {})
            mime = next((m for m in _MIME_PRIORITY if m in data), next(iter(data), None))
            if mime: blocks.append({'type':'display', 'mime':mime, 'data':data[mime]})
    return blocks

In [ ]:
#| export
def save_notebook(path:str|Path|None=None) -> Path:
    "Serialize `nb.cells` to a real Jupyter notebook file (`{nb.name}.ipynb` in the cwd, by default). Preserves each cell's original .ipynb id (or captures a freshly-assigned one) so unchanged cells don't produce noisy git diffs. The '#| export' pragma is prepended to code cell source here (and only here) -- nbdev's own parser needs it literally in the file, but boopiter keeps it out of c.source/the editor; see load_notebook() for the inverse."
    path = Path(path) if path else Path.cwd()/f'{nb.name}.ipynb'
    doc = _nbf.v4.new_notebook()
    for c in nb.cells:
        meta = {'boopiter': {'ctype': c.ctype, 'visible': c.visible, 'details': c.details}}
        kind = _BOOP2NB.get(c.ctype, 'raw')
        idkw = {'id': c.nb_id} if c.nb_id else {}
        src = f'#| export\n{c.source}' if (c.ctype == 'code' and c.export) else c.source
        if kind == 'code':
            outputs = _blocks_to_nb_outputs(c.output) if c.output else []
            cell = _nbf.v4.new_code_cell(src, outputs=outputs, metadata=meta, **idkw)
        elif kind == 'markdown':
            cell = _nbf.v4.new_markdown_cell(src, metadata=meta, **idkw)
        else:
            cell = _nbf.v4.new_raw_cell(src, metadata=meta, **idkw)
        c.nb_id = cell['id']  # capture the (possibly just-generated) id so future saves reuse it too
        doc.cells.append(cell)
    _nbf.write(doc, str(path))
    return path

## Loading

`load_notebook` is the inverse -- read an `.ipynb` back into `nb`, mapping nbformat cell types to boopiter's (`_NB_FALLBACK` handles plain, non-boopiter notebooks) and outputs back into blocks.

In [ ]:
#| export
_NB_FALLBACK = {'code':'code', 'markdown':'note', 'raw':'raw'}  # nbformat cell_type -> our ctype, for plain (non-boopiter) notebooks

In [ ]:
#| export
def load_notebook(path:str|Path) -> Notebook:
    "Load a Jupyter notebook file into `nb`, replacing its current contents. Inverse of save_notebook() -- detects a leading '#| export' line on code cells, sets c.export, and strips it out of the stored/displayed source."
    path = Path(path)
    doc = _nbf.read(str(path), as_version=4)
    nb.cells.clear()
    nb._nid = 0
    nb.selected = None
    for cell in doc.cells:
        meta = cell.get('metadata', {}).get('boopiter', {})
        ctype = meta.get('ctype')
        if ctype not in CTYPES + ('assistant',):
            ctype = _NB_FALLBACK.get(cell.cell_type, 'raw')  # plain (non-boopiter) notebook
        output = None
        if ctype == 'code':
            output = _nb_outputs_to_blocks(cell.get('outputs', [])) or None
        src, exported = cell.source, False
        if ctype == 'code' and _has_export(src):
            exported, src = True, _strip_export(src)
        nb.add(ctype, src, output=output, visible=meta.get('visible', True), nb_id=cell.get('id'),
               export=exported, details=meta.get('details'))
    nb.name = str(path.with_suffix(''))  # keep the directory, only strip .ipynb
    return nb

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()